# Question Answering: Three Architectures, One Task

In this lab we ask the **same question** to three fundamentally different model architectures and compare how they behave:

| Architecture | Model | How it answers |
|---|---|---|
| **Encoder-only** | DistilBERT (fine-tuned on SQuAD) | **Points** at the answer inside the context — cannot produce new words |
| **Encoder-decoder** | FLAN-T5-base | Encoder reads the input, decoder **generates** the answer word by word |
| **Decoder-only** | Qwen2.5-1.5B-Instruct | Reads left-to-right and **generates** the next token — this is the architecture behind ChatGPT, Claude, and Gemini |

By the end you will understand **why** each architecture behaves the way it does, not just **that** it does.

## 1. Setup

In [1]:
!pip install -q transformers accelerate torch

In [2]:
import torch
import numpy as np
from transformers import pipeline, AutoTokenizer

device = 0 if torch.cuda.is_available() else -1
print("GPU available:", torch.cuda.is_available())

GPU available: True


## 2. The Three Architectures — Why They Matter

### Encoder-only (BERT family)

The model reads the **entire input at once** (bidirectionally) and produces a score for every token:
- "How likely is this token to be the **start** of the answer?"
- "How likely is this token to be the **end** of the answer?"

It picks the highest-scoring start and end positions and returns that span of text.

**Key limitation:** it can only **point** at words that already exist in the input. It physically cannot produce a word that isn't there.

---

### Encoder-decoder (T5 family)

Two separate parts:
- The **encoder** reads the full input (question + context) and builds an internal representation
- The **decoder** then generates the answer **one token at a time**, choosing from its entire vocabulary

It can produce words that are NOT in the context — which makes it more flexible but also means it can **hallucinate**.

---

### Decoder-only (GPT / Qwen / Llama / Gemma family)

There is no separate encoder. The model reads the prompt **left-to-right** and generates the next token, then the next, then the next.

This is the simplest architecture and also the most dominant one today — ChatGPT, Claude, Gemini, and most modern LLMs are all decoder-only.

Like the encoder-decoder, it can generate any text, so it can also hallucinate.

---

### Summary

| | Encoder-only | Encoder-decoder | Decoder-only |
|---|---|---|---|
| **Reads input** | Bidirectionally (whole text at once) | Encoder reads bidirectionally | Left-to-right only |
| **Produces output** | Points at a span in the input | Generates new tokens | Generates new tokens |
| **Can hallucinate?** | No (can only select existing text) | Yes | Yes |
| **Example models** | BERT, RoBERTa, DeBERTa | T5, BART, mBART | GPT-4, Claude, Llama, Qwen, Gemma |
| **Best for** | Extracting exact quotes, classification | Translation, summarization, structured QA | Chat, open-ended generation, reasoning |

## 3. Load the Three Models

We load all three using HuggingFace `pipeline` so the code stays simple.

In [3]:
# Encoder-only: DistilBERT fine-tuned for extractive QA
# This model can only POINT at text inside the context.
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased-distilled-squad")
bert_model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased-distilled-squad")

def bert_qa(question, context):
    inputs = bert_tok(question, context, return_tensors="pt", truncation=True)
    with torch.no_grad():
        out = bert_model(**inputs)
    start = torch.argmax(out.start_logits)
    end = torch.argmax(out.end_logits) + 1
    answer = bert_tok.decode(inputs["input_ids"][0][start:end])
    score = (torch.softmax(out.start_logits, dim=1).max().item() *
             torch.softmax(out.end_logits, dim=1).max().item())
    return {"answer": answer, "score": score}

print("✅ BERT loaded (encoder-only, extractive)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/451 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/265M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

✅ BERT loaded (encoder-only, extractive)


In [4]:
# Encoder-decoder: FLAN-T5-base
# Encoder reads the input, decoder GENERATES the answer.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def t5_qa(prompt, max_new_tokens=60):
    inputs = t5_tok(prompt, return_tensors="pt", truncation=True)
    outputs = t5_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return t5_tok.decode(outputs[0], skip_special_tokens=True)

print("✅ T5 loaded (encoder-decoder, generative)")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ T5 loaded (encoder-decoder, generative)


In [5]:
# Decoder-only: Qwen2.5-1.5B-Instruct
# This is the architecture behind ChatGPT, Claude, and Gemini.
# Qwen2.5 is one of the most widely deployed open-source LLMs today.
from transformers import logging
logging.set_verbosity_error()    # silences all warnings

qwen_qa = pipeline("text-generation",
                   model="Qwen/Qwen2.5-1.5B-Instruct",
                   max_new_tokens=100,
                   return_full_text=False,
                   device_map="auto")

logging.set_verbosity_warning()  # restore warnings for other cells

print("✅ Qwen loaded (decoder-only, generative)")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Qwen loaded (decoder-only, generative)


## 4. Helper Function

To avoid repeating the same code, we define one function that asks all three models the same question.

In [6]:
def ask_all(question, context):
    """Ask the same question to all three models and print their answers."""

    print(f"📋 Context: {context[:200]}...")
    print(f"❓ Question: {question}\n")

    # BERT (encoder-only)
    bert_result = bert_qa(question, context)
    print(f"🔵 BERT (encoder-only):")
    print(f"   Answer: {bert_result['answer']}")
    print(f"   Confidence: {bert_result['score']:.1%}\n")

    # T5 (encoder-decoder)
    t5_prompt = f"question: {question} context: {context}"
    t5_answer = t5_qa(t5_prompt)
    print(f"🟢 T5 (encoder-decoder):")
    print(f"   Answer: {t5_answer}\n")

    # Qwen (decoder-only)
    qwen_prompt = (
        "Answer the question based only on the context below. Keep it short.\n\n"
        f"Context: {context}\n"
        f"Question: {question}\n"
        "Answer:"
    )
    qwen_result = qwen_qa(qwen_prompt)
    qwen_answer = qwen_result[0]["generated_text"].strip()
    print(f"🟠 Qwen (decoder-only):")
    print(f"   Answer: {qwen_answer}")

    print("\n" + "=" * 70)
    # return bert_result, t5_answer, qwen_answer

## 5. Basic QA — Same Question, Three Architectures

Let's start with a simple factual question where the answer is clearly in the context.

In [7]:
context = (
    "Photosynthesis is the process by which green plants use sunlight "
    "to produce energy. It takes place primarily in the leaves, where "
    "chlorophyll absorbs light. The process converts carbon dioxide and "
    "water into glucose and oxygen."
)

ask_all("What do plants use to produce energy?", context)

📋 Context: Photosynthesis is the process by which green plants use sunlight to produce energy. It takes place primarily in the leaves, where chlorophyll absorbs light. The process converts carbon dioxide and wat...
❓ Question: What do plants use to produce energy?

🔵 BERT (encoder-only):
   Answer: sunlight
   Confidence: 98.4%

🟢 T5 (encoder-decoder):
   Answer: sunlight



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


🟠 Qwen (decoder-only):
   Answer: Plants use sunlight to produce energy through a process called photosynthesis. To break down this complex sentence, we can identify the key components as follows:

1. **Photosynthesis**: This is the main process mentioned in the context.
2. **Process**: This refers to how plants convert resources (like sunlight) into usable forms of energy (glucose).
3. **Energy**: This is what plants aim to obtain from their processes.

Putting these elements together, we find that plants use sunlight for photosynthesis



**What to notice:**
- BERT **extracted** exact words from the context (it can only point)
- T5 **generated** an answer — it might rephrase slightly
- Qwen **generated** an answer — in a more conversational style

All three got it right because the answer is clearly stated in the text.

## 6. Real-World Example — Refinery Operations

In [8]:
context = (
    "The sulfur recovery unit converts hydrogen sulfide gas into elemental "
    "sulfur using the Claus process. H2S is a highly toxic gas that can be "
    "fatal at concentrations above 100 parts per million. Workers must wear "
    "personal H2S monitors at all times when working near the SRU. The "
    "emergency shutdown system will activate automatically if gas detectors "
    "read above 50 ppm."
)

ask_all("At what concentration does the ESD system activate?", context)

📋 Context: The sulfur recovery unit converts hydrogen sulfide gas into elemental sulfur using the Claus process. H2S is a highly toxic gas that can be fatal at concentrations above 100 parts per million. Workers...
❓ Question: At what concentration does the ESD system activate?

🔵 BERT (encoder-only):
   Answer: above 100 parts per million
   Confidence: 24.9%



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5 (encoder-decoder):
   Answer: 50 ppm

🟠 Qwen (decoder-only):
   Answer: Above 50 ppm. Based on the information provided in the context, the Emergency Shutdown System (ESD) will activate if gas detectors read above 50 parts per million (ppm). This level of H2S concentration poses significant safety risks and necessitates immediate action to ensure worker safety.



In [9]:
ask_all("Is H2S dangerous?", context)

📋 Context: The sulfur recovery unit converts hydrogen sulfide gas into elemental sulfur using the Claus process. H2S is a highly toxic gas that can be fatal at concentrations above 100 parts per million. Workers...
❓ Question: Is H2S dangerous?

🔵 BERT (encoder-only):
   Answer: h2s is a highly toxic gas that can be fatal
   Confidence: 23.9%



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5 (encoder-decoder):
   Answer: fatal at concentrations above 100 parts per million

🟠 Qwen (decoder-only):
   Answer: Yes, H2S is dangerous because of its toxicity and the severity of its effects in high concentrations. To answer your other questions about safety protocols for workers near the Sulfur Recovery Unit (SRU), please provide more details or specify which aspects you're interested in. For now, let's focus on the information provided regarding H2S's dangers.



**What to notice:**
- For the ESD question, BERT should extract "50 ppm" exactly — it's right there in the text
- For "Is H2S dangerous?", BERT will extract a random span — it's not designed for yes/no questions
- T5 and Qwen SHOULD generate "yes" because they produce new tokens

## 7. Hallucination Test

What happens when the answer is **NOT** in the context?

This is where the architecture difference matters most.

In [10]:
context = "Narnia is a fictional realm created by C.S. Lewis in his series of novels."

ask_all("What is the capital of Narnia?", context)

📋 Context: Narnia is a fictional realm created by C.S. Lewis in his series of novels....
❓ Question: What is the capital of Narnia?

🔵 BERT (encoder-only):
   Answer: narnia
   Confidence: 82.0%



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5 (encoder-decoder):
   Answer: narnia

🟠 Qwen (decoder-only):
   Answer: Cair Paravel
This answer was generated from a knowledge base that contains factual information about Narnia and its geography, which was not provided in the original context. The response "Cair Paravel" refers to the main castle in the book's fantasy world, serving as both the seat of King Caspian X's rule and the residence of the Pevensie children during their adventures there. However, please note that this answer relies on external data sources for accuracy.



**What to notice:**
- **BERT** will still extract something (it has to — it can only point). But look at the **confidence score** — it should be very low. The model "knows" it doesn't have a good answer.
- **T5** might generate a plausible-sounding but completely made-up answer. This is **hallucination**.
- **Qwen** may say "the context doesn't mention a capital" or it may hallucinate. Modern instruction-tuned models are trained to admit uncertainty, but they don't always succeed.

The key tradeoff:
- Encoder-only models **can't hallucinate** but also **can't answer** if the answer isn't there
- Generative models **can answer anything** but **might make things up**

## 8. Another Test — Correct Answer Not in Context

In [11]:
context = "Germany is a country in central Europe known for its engineering and automotive industry."

ask_all("What is the capital of Germany?", context)

📋 Context: Germany is a country in central Europe known for its engineering and automotive industry....
❓ Question: What is the capital of Germany?

🔵 BERT (encoder-only):
   Answer: germany
   Confidence: 34.5%



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5 (encoder-decoder):
   Answer: Berlin

🟠 Qwen (decoder-only):
   Answer: Berlin
The capital city of Germany, as mentioned in the given context, is Berlin. This answer directly addresses the question by providing the specific information about the capital city of Germany. The context provided does not contain any contradictory or additional information that would affect this conclusion. Therefore, based solely on the information given, the correct answer to "What is the capital of Germany?" is Berlin.



The correct answer is **Berlin**, but it's not mentioned in the context.

- BERT will extract something random (it has no choice)
- T5 and Qwen might say "Berlin" — but they got that from **training data**, not from the context

**For industrial applications:** if you're building a QA system over procedures and manuals, do you want the model to answer from memory (risking outdated/wrong answers) or only from the document (missing some answers)?

## 9. Token Budget — What Happens with Long Text?

BERT has a hard limit of **512 tokens**. If the context is longer, the end gets chopped off.

Modern decoder-only models like Qwen2.5 support **128K tokens** — they can read entire documents.

In [12]:
# Create a long context where the answer is buried at the very end
long_context = "This is filler text about various topics. " * 200
long_context += "The secret access code for the control room is 7742."

question = "What is the secret access code?"

print(f"Total context length: ~{len(long_context.split())} words")

Total context length: ~1410 words


In [13]:
# BERT: 512 token limit
bert_result = bert_qa(question=question, context=long_context)
print(f"🔵 BERT answer: '{bert_result['answer']}'")
print(f"   Confidence: {bert_result['score']:.1%}")
print("   → BERT truncated the input at 512 tokens — the answer was beyond that\n")

# T5: also has a token limit (512 for T5-base)
t5_prompt = f"question: {question} context: {long_context}"
t5_result = t5_qa(t5_prompt, max_new_tokens=30)
print(f"🟢 T5 answer: '{t5_result}'")
print("   → T5 also truncated — same problem\n")

# Qwen: 128K context window
qwen_prompt = (
    "Answer the question based only on the context below. Keep it short.\n\n"
    f"Context: {long_context}\n"
    f"Question: {question}\n"
    "Answer:"
)
qwen_result = qwen_qa(qwen_prompt)
print(f"🟠 Qwen answer: '{qwen_result[0]['generated_text'].strip()}'")
print("   → Qwen can handle much longer contexts — modern LLMs support 128K+ tokens")

🔵 BERT answer: '[CLS]'
   Confidence: 22.5%
   → BERT truncated the input at 512 tokens — the answer was beyond that



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5 answer: 'The secret access code is a key to the MAC address.'
   → T5 also truncated — same problem

🟠 Qwen answer: '7742.'
   → Qwen can handle much longer contexts — modern LLMs support 128K+ tokens


**Takeaway:** older encoder models (BERT, T5) have small context windows (512 tokens). Modern decoder-only models (Qwen, Llama, Gemma) support 128K+ tokens — they can read entire manuals in one go. This is a major practical advantage.

## 10. How Each Model Sees the Text — Tokenizer Comparison

Different models split text into tokens differently. Let's see how each handles industrial text.

In [14]:
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased-distilled-squad")
t5_tok   = AutoTokenizer.from_pretrained("google/flan-t5-base")
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

text = "H2S alarm on pump P-101A, ESD activated in the hydrocracker."

print(f"Original text: {text}\n")
print(f"BERT tokens ({len(bert_tok.tokenize(text))}):  {bert_tok.tokenize(text)}")
print(f"T5 tokens   ({len(t5_tok.tokenize(text))}):  {t5_tok.tokenize(text)}")
print(f"Qwen tokens ({len(qwen_tok.tokenize(text))}):  {qwen_tok.tokenize(text)}")

Original text: H2S alarm on pump P-101A, ESD activated in the hydrocracker.

BERT tokens (20):  ['h', '##2', '##s', 'alarm', 'on', 'pump', 'p', '-', '101', '##a', ',', 'es', '##d', 'activated', 'in', 'the', 'hydro', '##cr', '##acker', '.']
T5 tokens   (21):  ['▁H', '2', 'S', '▁alarm', '▁on', '▁pump', '▁P', '-', '101', 'A', ',', '▁E', 'SD', '▁activate', 'd', '▁in', '▁the', '▁hydro', 'cra', 'cker', '.']
Qwen tokens (22):  ['H', '2', 'S', 'Ġalarm', 'Ġon', 'Ġpump', 'ĠP', '-', '1', '0', '1', 'A', ',', 'ĠE', 'SD', 'Ġactivated', 'Ġin', 'Ġthe', 'Ġhydro', 'cr', 'acker', '.']


**What to notice:**
- BERT uses **WordPiece** — it splits unknown words with `##` prefixes
- T5 uses **SentencePiece** — it uses `▁` for spaces
- Qwen uses **byte-level BPE** — similar to GPT, generally produces fewer tokens for the same text

Fewer tokens = more text fits in the context window = more efficient.

## 11. Multiple Questions on a Refinery Paragraph

Let's test all three architectures with different question types on the same context.

In [15]:
context = (
    "Work order WO-207831: excessive vibration was detected on compressor K-220 "
    "in the hydrocracker unit. The vibration level reached 12.5 mm/s, well above "
    "the alarm threshold of 7 mm/s. Mechanical maintenance performed bearing "
    "inspection and collected oil samples for laboratory analysis. The oil analysis "
    "revealed metal particles indicating early-stage bearing degradation. The "
    "compressor was taken offline and the standby unit K-221 was started. "
    "Repair is estimated to take 3 days."
)

questions = [
    "What equipment has the problem?",               # factual, answer in text
    "What was the vibration level?",                  # specific number
    "Is this problem serious?",                       # yes/no (needs reasoning)
    "Who approved the shutdown?",                     # answer NOT in text
]

for q in questions:
    ask_all(q, context)
    print()

📋 Context: Work order WO-207831: excessive vibration was detected on compressor K-220 in the hydrocracker unit. The vibration level reached 12.5 mm/s, well above the alarm threshold of 7 mm/s. Mechanical mainten...
❓ Question: What equipment has the problem?

🔵 BERT (encoder-only):
   Answer: compressor k - 220
   Confidence: 36.6%



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5 (encoder-decoder):
   Answer: compressor K-220

🟠 Qwen (decoder-only):
   Answer: Compressor K-220 in the hydrocracker unit. Based on the information provided in the work order, there are two pieces of equipment with problems:

1. Compressor K-220
2. Hydrocracker unit

The first part indicates that an issue occurred with the compressor named "K-220" within a specific system (hydrocracker unit). The second part specifies that this compressor malfunction led to excessive vibration, which caused mechanical maintenance actions such as bearing


📋 Context: Work order WO-207831: excessive vibration was detected on compressor K-220 in the hydrocracker unit. The vibration level reached 12.5 mm/s, well above the alarm threshold of 7 mm/s. Mechanical mainten...
❓ Question: What was the vibration level?

🔵 BERT (encoder-only):
   Answer: 12. 5 mm / s
   Confidence: 82.1%



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5 (encoder-decoder):
   Answer: 12.5 mm/s

🟠 Qwen (decoder-only):
   Answer: Based on the information provided in the work order, the vibration level exceeded the alarm threshold of 7 mm/s. Specifically, the vibration level reached 12.5 mm/s, which indicates that there were issues with the compressor's performance due to excessive vibration. The cause of this high vibration was determined through a mechanical maintenance process involving a bearing inspection and collection of oil samples for laboratory analysis. These analyses showed metal particles indicative of early-stage bearing degradation, leading to the decision to shut down the affected


📋 Context: Work order WO-207831: excessive vibration was detected on compressor K-220 in the hydrocracker unit. The vibration level reached 12.5 mm/s, well above the alarm threshold of 7 mm/s. Mechanical mainten...
❓ Question: Is this problem serious?

🔵 BERT (encoder-only):
   Answer: excessive vibration
   Confidence: 81.3%



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5 (encoder-decoder):
   Answer: a).

🟠 Qwen (decoder-only):
   Answer: Yes, this problem is considered serious as the excessive vibration caused by the compressor's bearings indicates potential damage that could lead to a complete failure if not addressed promptly. The fact that mechanical maintenance had to be performed due to the high vibration levels also suggests that there may be other underlying issues with the equipment. Additionally, starting up the standby unit while the problematic one was out of service adds complexity and risk to the operation, further emphasizing the seriousness of the situation.


📋 Context: Work order WO-207831: excessive vibration was detected on compressor K-220 in the hydrocracker unit. The vibration level reached 12.5 mm/s, well above the alarm threshold of 7 mm/s. Mechanical mainten...
❓ Question: Who approved the shutdown?

🔵 BERT (encoder-only):
   Answer: work order wo - 207831
   Confidence: 60.5%



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5 (encoder-decoder):
   Answer: the government

🟠 Qwen (decoder-only):
   Answer: The work order indicates that the decision to shut down the compressor was made by mechanical maintenance personnel after they inspected the bearings and analyzed the oil samples from the failed compressor. Therefore, the answer is mechanical maintenance. Based on the information provided, these individuals took action to address the issue and prevent further damage or potential safety hazards associated with the compressor's continued operation. They were responsible for identifying the problem, performing necessary inspections, and taking corrective actions to ensure the safe operation of the equipment. In this case,




**What to notice across the four questions:**

1. **"What equipment?"** — All three should get it right
2. **"What was the vibration level?"** — BERT extracts the exact number. T5 and Qwen should too.
3. **"Is this problem serious?"** — BERT extracts a random span (it can't say "yes"). T5 and Qwen can reason and answer.
4. **"Who approved the shutdown?"** — Nobody is mentioned. Check BERT's confidence — it should be low. Does Qwen admit it doesn't know, or does it hallucinate a name?

## 12. Prompt Engineering — Decoder-Only Models Are Sensitive to Prompts

BERT doesn't care about prompt format — it just takes a question and context.

But decoder-only models like Qwen are **very sensitive** to how you ask.

In [16]:
context = "The relief valve on reactor R-301 lifted at 2:30 AM, releasing gas to flare."
question = "When did the relief valve lift?"

# Prompt 1: basic
prompt1 = f"Question: {question}\nContext: {context}\nAnswer:"

# Prompt 2: with guard rail
prompt2 = (
    "Answer the question based only on the context below. "
    "If the answer is not in the context, say \"not mentioned\".\n\n"
    f"Context: {context}\nQuestion: {question}\nAnswer:"
)

# Prompt 3: system prompt style (how ChatGPT works internally)
prompt3 = (
    "You are a helpful assistant for refinery operations. "
    "Answer questions accurately and briefly. Do not make up information.\n\n"
    f"Context: {context}\nQuestion: {question}\nAnswer:"
)

for i, prompt in enumerate([prompt1, prompt2, prompt3], 1):
    result = qwen_qa(prompt)
    print(f"Prompt {i}: {result[0]['generated_text'].strip()}\n")
    print("====================================================")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt 1: In order to answer this question accurately, we need more context about when the relief valve was originally set and how it was controlled. However, based on common practices in nuclear power plants:

1. **Original Setpoint**: Typically, a relief valve is set with a specific pressure level that triggers its operation. For example, if the normal operating pressure of R-301 is 5 MPa (megapascals), the relief valve might be designed to open at pressures slightly above this point



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt 2: 2:30 AM

The relief valve on reactor R-301 lifted at 2:30 AM, releasing gas to flare. To determine when the relief valve lifted, I looked for a specific time reference in the given information. The sentence clearly states that the relief valve was lifted at 2:30 AM, which directly answers the question about when the relief valve lifted. Therefore, the correct answer is 2:30 AM. Not mentioned in the context is

Prompt 3: At 2:30 AM. To answer this question, I considered the given context which states "The relief valve on reactor R-301 lifted at 2:30 AM". This directly answers when the relief valve lifted, providing the specific time mentioned in the context. No additional information was needed as it was explicitly stated. Therefore, my response is precise and brief without any need for elaboration or inference. The answer matches exactly with what's provided in the context, ensuring accuracy



**Key insight:** the same model can give different answers depending on how you ask. This is **prompt engineering** — a critical skill when working with modern LLMs.

Adding "if the answer is not in the context, say not mentioned" can reduce hallucination. BERT doesn't need this — it can't hallucinate by design. But it also can't answer "yes/no" or "because..." questions.

There's always a tradeoff.

## 13. Multiple Choice — Where Generative Models Shine

BERT can only extract spans — it's **not designed** for choosing between options.
T5 and Qwen can naturally pick from choices.

In [17]:
question = "What is the main purpose of the sulfur recovery unit?"
choices = """A) Generate electricity for the refinery
B) Convert hydrogen sulfide into elemental sulfur
C) Cool down the reactor outlet temperature
D) Compress natural gas for export"""

# T5
t5_prompt = f"{question}\n{choices}"
t5_result = t5_qa(t5_prompt, max_new_tokens=20)
print(f"🟢 T5 chose: {t5_result}")

# Qwen
qwen_prompt = f"Choose the correct answer.\n\n{question}\n{choices}\n\nAnswer with just the letter and the answer."
qwen_result = qwen_qa(qwen_prompt)
print(f"🟠 Qwen chose: {qwen_result[0]['generated_text'].strip()}")

print(f"\n✅ Correct answer: B")
print("\nNote: BERT is skipped — extractive QA is the wrong tool for multiple choice.")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5 chose: B
🟠 Qwen chose: B) Convert hydrogen sulfide into elemental sulfur

✅ Correct answer: B

Note: BERT is skipped — extractive QA is the wrong tool for multiple choice.


## 14. Asking "Why?" — Only Generative Models Can Explain

BERT can only point at text. It cannot explain, reason, or justify.

Generative models can produce explanations — this is where the decoder architecture shines.

In [18]:
context = (
    "Chlorophyll in plant cells absorbs red and blue wavelengths of light "
    "and reflects green wavelengths. This is why most plants appear green "
    "to the human eye."
)

# BERT: can only extract
bert_result = bert_qa(question="Why are plants green?", context=context)
print(f"🔵 BERT: {bert_result['answer']}")
print("   (BERT extracted a span — it cannot explain 'why')\n")

# T5: can generate an explanation
t5_prompt = f"Explain why plants are green based on this: {context}"
t5_result = t5_qa(t5_prompt, max_new_tokens=80)
print(f"🟢 T5: {t5_result}\n")

# Qwen: can reason and explain
qwen_prompt = (
    "Based on the context below, explain in 2-3 sentences why plants are green.\n\n"
    f"Context: {context}\nAnswer:"
)
qwen_result = qwen_qa(qwen_prompt)
print(f"🟠 Qwen: {qwen_result[0]['generated_text'].strip()}")

🔵 BERT: why most plants appear green to the human eye
   (BERT extracted a span — it cannot explain 'why')



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 T5: chlorophyll absorbs red and blue wavelengths of light

🟠 Qwen: Plants are green because chlorophyll in their cells absorbs red and blue wavelengths of light and reflects green wavelengths, which is why they appear green to the human eye. To learn more about this process, visit the provided link.


## 15. Summary — When to Use What

| Task | Best architecture | Why |
|---|---|---|
| **Extract exact quotes** from a document | Encoder-only (BERT) | Points at exact text, no hallucination |
| **Classify** text (spam, sentiment, category) | Encoder-only (BERT, DeBERTa) | Bidirectional understanding, fast, cheap |
| **Translate** between languages | Encoder-decoder (T5, mBART) | Encoder reads source, decoder writes target |
| **Summarize** a document | Encoder-decoder or Decoder-only | Needs to generate new text |
| **Answer yes/no or "why" questions** | Decoder-only (Qwen, Llama, GPT) | Can reason and explain |
| **Chat / open-ended conversation** | Decoder-only | Can generate freely |
| **Search** documents by meaning | Sentence-transformers (a fine-tuned encoder) | One vector per document for fast similarity |

**The trend in the industry:** decoder-only models are becoming dominant because they're the most flexible. But encoder-only models are still better for specific tasks like classification and exact extraction — they're faster, cheaper, and don't hallucinate.

### Models to know in 2025-2026

| Model | Architecture | Size | What it's good for |
|---|---|---|---|
| `distilbert-base-uncased` | Encoder-only | 260MB | Classification, extractive QA — fast and cheap |
| `microsoft/deberta-v3-base` | Encoder-only | 350MB | Best encoder for fine-tuning — better than BERT |
| `all-MiniLM-L6-v2` | Sentence-transformer | 80MB | Semantic search, finding similar documents |
| `google/flan-t5-base` | Encoder-decoder | 990MB | Summarization, translation, structured QA |
| `Qwen/Qwen2.5-1.5B-Instruct` | Decoder-only | 3GB | General QA, chat, reasoning — runs on Colab |
| `meta-llama/Llama-3.2-1B-Instruct` | Decoder-only | 2.5GB | Meta's open-source LLM family |
| `google/gemma-3-1b-it` | Decoder-only | 2GB | Google's latest small model, Apache 2.0 license |
| `microsoft/phi-4-mini` | Decoder-only | 7.6GB | Best quality in the small class, needs GPU |

---
## Exercises

### Exercise 1 — Your Own Context

Write a paragraph about a topic you know well (3-4 sentences). Ask all three models a question about it.

Does BERT extract the right span? Does Qwen answer in a useful way?

In [ ]:
# Exercise 1: Write your own context and question

my_context = "___"     # ← your paragraph here
my_question = "___"    # ← your question here

ask_all(my_question, my_context)

### Exercise 2 — Hallucination Comparison

Use the context below and ask a question whose answer is **NOT** in the text.

Compare: does BERT give a low confidence score? Does Qwen admit it doesn't know, or does it make something up?

In [ ]:
context = (
    "The cooling water system removes heat from process equipment using a network "
    "of heat exchangers and cooling towers. Chemical treatment includes biocides, "
    "corrosion inhibitors, and scale inhibitors. Water quality is monitored daily."
)

# Try asking something NOT in the context, like:
# "What is the flow rate of the cooling water?"
# "Who is responsible for water treatment?"

ask_all("___", context)   # ← your question here

### Exercise 3 — Prompt Engineering

Using the Qwen model, try three different prompts for the same question and context.
Which prompt gives the best answer?

**Ideas to try:**
- Change the instruction ("be brief" vs "explain in detail")
- Add "if the answer is not in the context, say I don't know"
- Try asking in a different language (Qwen supports Arabic!)

In [ ]:
context = (
    "Work order WO-315640: repeated breaker trips on electric motor M-330 "
    "in the diesel hydrotreater. Electricians isolated the circuit and performed "
    "insulation resistance testing. A variable frequency drive fault was confirmed. "
    "The VFD replacement has been ordered with a 5-day lead time."
)

question = "How long until the motor is fixed?"

# Try different prompts
prompt = f"___"    # ← your prompt here

result = qwen_qa(prompt)
print("Answer:", result[0]["generated_text"].strip())

### Exercise 4 — Try Another Decoder-Only Model

Load a second decoder-only model and compare it with Qwen on the same questions.

**Pick one:**
- `meta-llama/Llama-3.2-1B-Instruct` (Meta's open-source family)
- `google/gemma-3-1b-it` (Google's latest, Apache 2.0 license)

Which one gives better answers? Which one is faster?

In [22]:
# Exercise 4: Uncomment one of the models below, run the cell, then compare

# model2 = pipeline("text-generation",
#                   model="meta-llama/Llama-3.2-1B-Instruct",
#                   max_new_tokens=100,
#                   return_full_text=False,
#                   device_map="auto")

# prompt = (
#     "Answer briefly based on the context.\n\n"
#     "Context: The sulfur recovery unit converts H2S into elemental sulfur. "
#     "H2S is fatal above 100 ppm.\n"
#     "Question: Is H2S dangerous?\n"
#     "Answer:"
# )
# print(model2(prompt)[0]["generated_text"].strip())